# MIDL quick demo for math problem

Build `D_in` -> compute `basis` -> construct dimensionless `Pi` -> run `MIDL.fit` -> recover exponents.

In [23]:
import numpy as np
import importlib
import midl

# Ensure notebook uses the latest local midl.py changes
importlib.reload(midl)
MIDL = midl.MIDL


In [19]:
# Synthetic data
rng = np.random.default_rng(123)
n_samples = 320

x = rng.uniform(0.1, 3.24, n_samples)
y = rng.uniform(0.1, 3.24, n_samples)
z = rng.uniform(0.1, 3.24, n_samples)
L = rng.uniform(0.1, 1.1, n_samples)

X_raw = np.column_stack([x, y, z, L])
u = np.sin(x*y/L/L) * np.cos(x*z/L/L)
# Dimension matrix D_in (shape: (l, m))
D_in = np.matrix([1, 1, 1, 1])
print("Dimension matrix D_in:\n", D_in)

# Compute basis vectors (π exponents) and rank
basis, r = midl.calc_basis(D_in)
print("Null space dimension:", D_in.shape[1] - r)
print("basis shape:", basis.shape)
print("basis (π exponents):\n", basis)

# Transform X_raw to dimensionless inputs Pi
Pi = np.exp(np.log(X_raw) @ basis)
#print("Pi shape:", Pi.shape)


Dimension matrix D_in:
 [[1 1 1 1]]
Null space dimension: 3
basis shape: (4, 3)
basis (π exponents):
 [[-0.5        -0.5        -0.5       ]
 [ 0.83333333 -0.16666667 -0.16666667]
 [-0.16666667  0.83333333 -0.16666667]
 [-0.16666667 -0.16666667  0.83333333]]


In [20]:
# Run MIDL
model = MIDL(
    k_neighbors=6,
    de_maxiter=200,
    de_popsize=15,
    random_state=42,
)

result = model.fit(Pi_independent=Pi, pi_dependent=u)
pi_hat = MIDL.compose_new_pi(Pi, result["W"])

print("\n=== MIDL Results ===")
print("MI scores:", result["mi_scores"])
#print("W (columns are w_i):")
#print(result["W"])
print("dominant_q:", result["dominant_q"])
print("drop ratios I_i / I_(i+1):", result["drop_ratios"])

# Recover exponents in the original variable space
alpha = basis @ result["W"]
print("\n=== Recovered exponents ===")
print(alpha)


[Step 1] MI = 0.149088
[Step 2] MI = 0.082720
   ratio = 1.802
[Step 3] MI = 0.029027
   ratio = 2.850

=== MIDL Results ===
MI scores: [0.14908846 0.08271989 0.02902672]
dominant_q: 3
drop ratios I_i / I_(i+1): None

=== Recovered exponents ===
[[ 0.41353944  0.28191985  0.70675762]
 [ 0.00973962  0.61847404 -0.60613117]
 [ 0.39635212 -0.70773964 -0.30333084]
 [-0.81963118 -0.19265425  0.2027044 ]]


In [21]:
if result["dominant_q"] >= 2:
    ax2d, ax3d = MIDL.plot_component_vs_dependent(
        Pi_independent=Pi,
        pi_dependent=u,
        W=result["W"],
        dominant_q=result["dominant_q"],
        component_index=0,
    )
else:
    ax2d = MIDL.plot_component_vs_dependent(
        Pi_independent=Pi,
        pi_dependent=u,
        W=result["W"],
        dominant_q=result["dominant_q"],
        component_index=0,
    )

TypeError: MIDL.plot_component_vs_dependent() got an unexpected keyword argument 'dominant_q'